# Data Enrichment

This notebook enriches the acquired homestay dataset by:

1. Generating price information.
2. Calculating distances from major tourist attractions.
3. Creating proximity labels.
4. Generating synthetic homestay descriptions using Qwen.

The enriched dataset will be used for feature engineering and recommendation generation.

In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

# Data manipulation
import re
from difflib import SequenceMatcher

import pandas as pd
import numpy as np

from geopy.distance import geodesic

In [15]:
# ==========================================
# LOAD ACQUIRED DATASET
# ==========================================

df = pd.read_csv(
    "../data/processed/homestays_acquired.csv"
)

print(df.shape)

df.head()

(1157, 15)


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,google_address,latitude,longitude,rating,review_count
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,KALIMPONG,Municipality,"8th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,"3F76+75W, Rishi Rd, Mongbol Busty, Kalimpong, ...",27.062761,88.460383,5.0,1
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,KALIMPONG,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,"Kalimpong, West Bengal, Chandraloke, Kalimpong...",27.052709,88.461699,4.2,77
2,3,BETHANY HOMESTAY,ANUPAMA TAMANG,Silver,KALIMPONG,Kalimpong 1,Dr.GRAHAMS HOME BLOCK B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,"Unnamed Road, Deolo, Dalapchan Ridge Reserve F...",27.082570,88.509402,4.6,69
3,4,S3 HOMESTAY,SANGITA RAI,Silver,KALIMPONG,Kalimpong 1,UPPER ECHHEY DARA GAON \r\nKALIMPONG,sangitasankalp@gmail.com,9933410313,Sunrise Inn Homestay,"E Main Rd, Chandraloke, Kalimpong, West Bengal...",27.053167,88.467557,4.3,106
4,5,BAJARANGI HOMESTAY,KAMAL KUMAR SHARMA,Silver,KALIMPONG,Kalimpong 1,SINGI SAMALBONG KALIMPONG,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,"Sinji, Sukrabarey Bazar, Kalimpong, Samalbong ...",27.008221,88.512633,4.2,5


## Identity Verification & Drop Unverified Records

`01_data_acquisition.ipynb` is purely acquisition -- it does not
classify match confidence. That audit happens here instead, before any
enrichment work is done on records that would end up excluded anyway.

Each record is classified using two independent signals:

1. **Name similarity** between `homestay_name` and `google_name`.
2. **Address plausibility** -- whether `google_address` contains a token
   from the homestay's own `village`/`block`.

A high name-similarity score alone is trusted. A lower score is only
trusted if the address independently corroborates it (e.g. a homestay
correctly matched under a different trading name on Google). Records
with neither signal are excluded before any further processing.

In [16]:
# ==========================================
# NAME SIMILARITY
# ==========================================

def name_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    return SequenceMatcher(None, str(a).lower().strip(), str(b).lower().strip()).ratio()

df["name_match_score"] = df.apply(
    lambda row: name_similarity(row["homestay_name"], row["google_name"]),
    axis=1,
)


# ==========================================
# ADDRESS / LOCATION PLAUSIBILITY CHECK
# ==========================================

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def location_plausible(row):
    address = normalize_text(row.get("google_address"))
    village = normalize_text(row.get("village"))
    block = normalize_text(row.get("block"))

    if not address:
        return False

    stopwords = {"kalimpong", "west", "bengal", "india", "block", "gaon", "busty"}
    tokens = set(
        [w for w in village.split() if len(w) > 2 and w not in stopwords]
        + [w for w in block.split() if len(w) > 2 and w not in stopwords]
    )

    if not tokens:
        return None

    return any(t in address for t in tokens)

df["location_plausible"] = df.apply(location_plausible, axis=1)


# ==========================================
# COMBINED MATCH-STATUS CLASSIFICATION
# ==========================================

def classify(row):
    name_score = row["name_match_score"]
    loc_ok = row["location_plausible"]

    if pd.isna(name_score):
        return "no_match"
    if name_score >= 0.75:
        return "high_confidence"
    if loc_ok is True:
        return "likely_renamed"
    elif loc_ok is False:
        return "likely_wrong"
    else:
        return "needs_manual_review"

df["match_status"] = df.apply(classify, axis=1)

print(df["match_status"].value_counts())
print()
print(df["match_status"].value_counts(normalize=True).mul(100).round(1))


# ==========================================
# SELECTION-BIAS CHECK
# ==========================================

kept_mask = df["match_status"].isin(["high_confidence", "likely_renamed"])

print("\nCategory distribution -- kept records:")
print(df.loc[kept_mask, "category"].value_counts(normalize=True).round(3))

print("\nCategory distribution -- dropped records:")
print(df.loc[~kept_mask, "category"].value_counts(normalize=True).round(3))


# ==========================================
# SAVE FULL AUDIT TRAIL (for appendix / reproducibility)
# ==========================================

df.to_csv("../data/processed/homestays_acquired_full_audit.csv", index=False)


# ==========================================
# DROP UNVERIFIED RECORDS
# ==========================================

before = len(df)

df = df[kept_mask].copy().reset_index(drop=True)
df = df.drop(columns=["name_match_score", "location_plausible", "match_status"])

after = len(df)
print(f"\nKept {after} of {before} records ({after/before*100:.1f}%).")
print(f"Dropped {before - after} unverified records.")

match_status
high_confidence    614
likely_wrong       344
likely_renamed     199
Name: count, dtype: int64

match_status
high_confidence    53.1
likely_wrong       29.7
likely_renamed     17.2
Name: proportion, dtype: float64

Category distribution -- kept records:
category
Silver    0.789
silver    0.109
SILVER    0.072
Gold      0.020
gold      0.004
GOLD      0.004
sliver    0.001
Siver     0.001
Name: proportion, dtype: float64

Category distribution -- dropped records:
category
Silver     0.801
silver     0.097
SILVER     0.079
Gold       0.012
Siver      0.006
GOLD       0.003
SILVRER    0.003
Name: proportion, dtype: float64

Kept 813 of 1157 records (70.3%).
Dropped 344 unverified records.


In [ ]:
# ==========================================
# STANDARDIZE CATEGORY 
# ==========================================

before_missing = df["category"].isna().sum()

df["category"] = df["category"].fillna("Silver")
df["category"] = df["category"].astype(str).str.strip().str.title()

typo_corrections = {
    "Siver": "Silver",
    "Sliver": "Silver",
    "Silvrer": "Silver",
}
df["category"] = df["category"].replace(typo_corrections)

print(f"Category values imputed as 'Silver': {before_missing}")
print(df["category"].value_counts())

Category values imputed as 'Silver': 18
category
Silver    791
Gold       22
Name: count, dtype: int64


In [18]:
# ==========================================
# TOURIST LOCATIONS
# ==========================================

TOURIST_LOCATIONS = {
    "deolo": (27.08928, 88.50333), 
    "durpin": (27.03609, 88.45346), 
    "town": (27.05959, 88.46943), 
    "lava": (27.15992, 88.60598), 
    "pedong": (27.15945, 88.61551), 
    "gorubathan": (26.95435, 88.69557), 
    "rishop": (27.11166, 88.65187), 
    "lolegaon": (27.01909, 88.56538) 
}

In [19]:
# ==========================================
# DISTANCE CALCULATION FUNCTION
# ==========================================

def distance_km(
    lat,
    lon,
    target_lat,
    target_lon
):

    if pd.isna(lat) or pd.isna(lon):
        return None

    return round(
        geodesic(
            (lat, lon),
            (target_lat, target_lon)
        ).km,
        2
    )

In [20]:
# ==========================================
# DISTANCE FEATURE GENERATION
# ==========================================

for location_name, (target_lat, target_lon) in TOURIST_LOCATIONS.items():

    column_name = f"distance_to_{location_name}"

    df[column_name] = df.apply(
        lambda row: distance_km(
            row["latitude"],
            row["longitude"],
            target_lat,
            target_lon
        ),
        axis=1
    )

print("Distance features generated successfully.")

Distance features generated successfully.


In [21]:
# ==========================================
# PROXIMITY LABELS
# ==========================================

def proximity(distance):

    if pd.isna(distance):
        return None

    elif distance <= 5:
        return "Very Close"

    elif distance <= 10:
        return "Moderately Close"

    else:
        return "Far"

In [ ]:
# ==========================================
# PROXIMITY FEATURE GENERATION
# ==========================================

for location_name in TOURIST_LOCATIONS.keys():
    df[f"{location_name}_proximity"] = (
        df[f"distance_to_{location_name}"]
        .apply(proximity)
    )

In [23]:
# ==========================================
# PRICE GENERATION
# ==========================================

np.random.seed(42)

def generate_price(row):

    if str(row["category"]).upper() == "GOLD":
        return np.random.randint(2500, 4501)

    return np.random.randint(1000, 2201)

df["price"] = df.apply(
    generate_price,
    axis=1
)

df["price"].describe()

count     813.000000
mean     1680.364084
std       500.269684
min      1000.000000
25%      1343.000000
50%      1675.000000
75%      1935.000000
max      4466.000000
Name: price, dtype: float64

In [24]:
# ==========================================
# AMENITIES FEATURE GENERATION
# ==========================================

np.random.seed(42)  # reset so this cell is reproducible even if re-run alone

def assign_amenities(row):

    if row["category"] == "Gold":

        return pd.Series({
            "wifi": np.random.choice([0,1], p=[0.1,0.9]),
            "parking": np.random.choice([0,1], p=[0.15,0.85]),
            "breakfast": np.random.choice([0,1], p=[0.1,0.9]),
            "mountain_view": np.random.choice([0,1], p=[0.3,0.7]),
            "room_service": np.random.choice([0,1], p=[0.4,0.6]),
            "bonfire_barbeque": np.random.choice([0,1], p=[0.5,0.5]),
            "pickup_dropoff_service": np.random.choice([0,1], p=[0.6,0.4])
        })

    else:

        return pd.Series({
            "wifi": np.random.choice([0,1], p=[0.3,0.7]),
            "parking": np.random.choice([0,1], p=[0.25,0.75]),
            "breakfast": np.random.choice([0,1], p=[0.2,0.8]),
            "mountain_view": np.random.choice([0,1], p=[0.4,0.6]),
            "room_service": np.random.choice([0,1], p=[0.75,0.25]),
            "bonfire_barbeque": np.random.choice([0,1], p=[0.6,0.4]),
            "pickup_dropoff_service": np.random.choice([0,1], p=[0.8,0.2])
        })

df[[
    "wifi",
    "parking",
    "breakfast",
    "mountain_view",
    "room_service",
    "bonfire_barbeque",
    "pickup_dropoff_service"
]] = df.apply(assign_amenities, axis=1)

In [25]:
# ==========================================
# VALIDATION CHECKS (before saving)
# ==========================================

print("Price distribution by category:")
print(df.groupby("category")["price"].describe()[["mean", "min", "max"]])

print("\nAmenity rates by category:")
amenity_cols = ["wifi", "parking", "breakfast", "mountain_view",
                 "room_service", "bonfire_barbeque", "pickup_dropoff_service"]
print(df.groupby("category")[amenity_cols].mean().round(2))

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

Price distribution by category:
                 mean     min     max
category                             
Gold      3768.818182  2534.0  4466.0
Silver    1622.278129  1000.0  2199.0

Amenity rates by category:
          wifi  parking  breakfast  mountain_view  room_service  \
category                                                          
Gold      0.95     0.82       0.82           0.64          0.64   
Silver    0.71     0.76       0.79           0.58          0.25   

          bonfire_barbeque  pickup_dropoff_service  
category                                            
Gold                  0.41                    0.27  
Silver                0.37                    0.20  

Missing values:
owner_email    16
dtype: int64


In [26]:
# ==========================================
# SAVE ENRICHED DATASET
# ==========================================

df.to_csv(
    "../data/processed/homestays_enriched.csv",
    index=False
)

print(
    "Enriched dataset saved successfully."
)

Enriched dataset saved successfully.
